In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

def predict_future_demand(model_data, days=14):
    """预测未来需求"""
    last_date = model_data['date'].max()
    future_dates = [last_date + timedelta(days=i) for i in range(1, days+1)]
    
    historical_national = model_data.groupby('date').agg({'qty_alipay': 'sum'}).reset_index()
    historical_national['dayofweek'] = historical_national['date'].dt.dayofweek
    historical_national['month'] = historical_national['date'].dt.month
    
    weekday_pattern = historical_national.groupby('dayofweek')['qty_alipay'].mean()
    monthly_pattern = historical_national.groupby('month')['qty_alipay'].mean()
    regional_ratio = model_data.groupby('region_id')['qty_alipay'].sum() / model_data['qty_alipay'].sum()
    
    national_forecast = []
    for i, future_date in enumerate(future_dates):
        dayofweek = future_date.weekday()
        month = future_date.month
        
        base_demand = (weekday_pattern.get(dayofweek, historical_national['qty_alipay'].mean()) + 
                      monthly_pattern.get(month, historical_national['qty_alipay'].mean())) / 2
        
        trend_factor = 1.0 + (i * 0.01)
        seasonal_factor = 1.2 if month in [11, 12] else (0.9 if month in [1, 2] else 1.0)
        
        predicted_demand = max(0, base_demand * trend_factor * seasonal_factor)
        national_forecast.append({
            'date': future_date,
            'predicted_demand': predicted_demand,
            'dayofweek': dayofweek,
            'month': month
        })
    
    national_forecast_df = pd.DataFrame(national_forecast)
    
    # 区域预测分配
    regional_forecast = []
    for region_id, ratio in regional_ratio.items():
        regional_daily = []
        for _, national_row in national_forecast_df.iterrows():
            regional_daily.append({
                'date': national_row['date'],
                'region_id': region_id,
                'predicted_demand': national_row['predicted_demand'] * ratio
            })
        regional_df = pd.DataFrame(regional_daily)
        regional_forecast.append({
            'region_id': region_id,
            'total_predicted_demand': regional_df['predicted_demand'].sum(),
            'daily_predictions': regional_df
        })
        
    return national_forecast_df, regional_forecast